# Can Abnormal IoT Traffic Be Detected?
### Behavioral Detection of Botnet-Infected IoT Devices

**Computer Security Project — Data-Driven Security Analysis**

| | |
|---|---|
| **Dataset** | N-BaIoT — [Kaggle](https://www.kaggle.com/datasets/mkashifn/nbaiot-dataset) |
| **Research question** | *Can behavioral network-traffic features accurately distinguish benign IoT traffic from botnet-infected IoT traffic?* |
| **Environment** | Google Colab / Python 3 · scikit-learn · XGBoost · SHAP |
| **Author** | _(add your name)_ |

**Notebook map**

| § | Rubric step |
|---|---|
| 1 | Step 1 — Select a Kaggle Dataset |
| 2 | Step 2 — Define a Security Problem |
| 3 | Step 3 — Data Exploration |
| 4 | Step 4 — Data Preprocessing |
| 5 | Step 5 — Security Analysis / Machine Learning |
| 6 | Step 6 — Model Evaluation |
| 7 | Step 7 — Security Interpretation |
| 8 | Optional advanced component — SHAP / feature importance |

> **Academic integrity:** all code, analysis, and visualizations in this notebook were written for this project. The dataset and the original N-BaIoT paper are cited in §References.

### Code provenance — cite external code/libraries

Every line of modelling, evaluation and plotting code in this notebook was written for this project; no external notebook or solution code was copied. The work does, however, depend on third-party libraries, which are cited here and in the report bibliography:

| Library | Used for | Citation |
|---|---|---|
| `scikit-learn` | preprocessing, Logistic Regression, Random Forest, Isolation Forest, metrics | Pedregosa *et al.* (2011) |
| `xgboost` | gradient-boosted trees | Chen & Guestrin (2017) |
| `shap` | post-hoc feature attribution | Lundberg & Lee (2017) |
| `pandas` / `numpy` | data handling | van Rossum (2007); Harris *et al.* (2020) |
| `matplotlib` / `seaborn` | plotting | Hunter (2007); Waskom (2021) |

Dataset: **N-BaIoT**, Kashif (2020), https://www.kaggle.com/datasets/mkashifn/nbaiot-dataset — with thanks to Miramirkhani *et al.* (2017), who collected the traffic.


In [ ]:
# ============================================================
#  §0  Environment, configuration, reproducibility
# ============================================================
import os, glob, random, warnings, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')

SEED        = 42
TARGET_ROWS = 150_000   # stratified sample size -> notebook runs in minutes
TEST_SIZE   = 0.20
VARIANCE_EPS = 1e-9     # near-zero-variance threshold
CORR_THRESH  = 0.95     # feature-pruning correlation threshold

random.seed(SEED)
np.random.seed(SEED)

print('OK  | seed =', SEED, '| target rows =', f'{TARGET_ROWS:,}')

---
## §1  Select a Kaggle Dataset

**Dataset name:** N-BaIoT Dataset to Detect IoT Botnet Attacks  
**Kaggle URL:** <https://www.kaggle.com/datasets/mkashifn/nbaiot-dataset>  
**Citation:** Kashif, M. (2020). *N-BaIoT Dataset to Detect IoT Botnet Attacks* [Data set]. Kaggle.  
**Underlying study:** Meidan, Y. et al. (2018). *N-BaIoT: Network-based Detection of IoT Botnet Attacks Using Deep Autoencoders*. arXiv:1805.03409.

**Description.** Real network traffic was captured from nine commercial IoT devices connected over Wi-Fi to an enterprise-style lab network. Each device was first observed under normal operation and then authentically infected with the **Mirai** and **BASHLITE (Gafgyt)** botnets, which executed scanning, junk, UDP, TCP, ACK and SYN flood attacks. Rather than raw packets, every record is a *behavioral snapshot*: 115 statistical features describing packet sizes, packet counts, and packet jitter, aggregated by source IP, source MAC-IP, channel and socket across five time windows (100 ms, 500 ms, 1.5 s, 10 s, 1 min).

**Target variable:** a binary label constructed below — `0 = benign`, `1 = malicious`.

**Important features** (full definitions in §3):

| Group | Statistics | Why it matters for security |
|---|---|---|
| Packet size (outbound) | mean, variance | Flooding replaces normal payloads with large or uniform junk packets |
| Packet count | count | Scanning and floods drive packet rates far above baseline |
| Packet jitter | mean, variance, number | Bot command-and-control beacons and flood generators alter inter-arrival timing |
| Packet size (in+out) | magnitude, radius, covariance, correlation | Captures the shift from request/response traffic to one-way attack traffic |

Below we locate the dataset and build the raw file inventory. The N-BaIoT release stores **one CSV per device per traffic type**, so the label must be constructed from the file names.

In [ ]:
# ============================================================
#  Locate the N-BaIoT CSV files.
#  Tries (1) Kaggle/Colab working dirs, (2) local folder, (3) zip, (4) upload.
# ============================================================
CANDIDATE_ROOTS = [
    '/kaggle/input/nbaiot-dataset', 'nbaiot-dataset', 'N-BaIoT',
    'data', 'dataset', '.',
]

def first_csv_root(roots):
    for r in roots:
        p = Path(r)
        if p.exists() and any(p.rglob('*.csv')):
            return str(p)
    return None

DATA_ROOT = first_csv_root(CANDIDATE_ROOTS)

# Fallback 1: a zip sitting next to the notebook
if DATA_ROOT is None:
    zips = list(Path('.').glob('*.zip'))
    if zips:
        z = zips[0]
        print('Extracting', z.name)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(z.stem)
        DATA_ROOT = first_csv_root([z.stem])

# Fallback 2: interactive upload (works in Google Colab / Kaggle)
if DATA_ROOT is None:
    try:
        from google.colab import files as _files
        print('Upload the N-BaIoT zip (or a zip containing the CSV folders):')
        up = _files.upload()
        name = next(iter(up))
        with zipfile.ZipFile(name) as zf:
            zf.extractall(Path(name).stem)
        DATA_ROOT = first_csv_root([Path(name).stem])
    except Exception as e:
        print('Upload unavailable:', e)

assert DATA_ROOT, 'Dataset not found - upload nbaiot-dataset.zip and re-run.'
print('DATA_ROOT =', DATA_ROOT)
print('CSV files found:', len(list(Path(DATA_ROOT).rglob('*.csv'))))

In [ ]:
# ============================================================
#  Build the file inventory, then load a stratified sample.
#  Sampling is done with skiprows so the full 7M-row corpus never
#  has to fit in memory (the notebook stays Colab-friendly).
# ============================================================

BENIGN_KEYS  = ('benign', 'normal', 'clean', 'legit')
BOTNET_KEYS  = ('mirai', 'gafgyt', 'bashlite')   # note: gafgyt == bashlite

def classify_file(path: Path):
    """Infer (label, attack_type, device) from a CSV's position in the tree.

    Handles the layouts seen in the wild:
      <device>/benign_traffic.csv    label comes from the file name
      <device>/mirai/scan.csv        botnet comes from the enclosing folder and
      <device>/gafgyt/junk.csv       the device from the top-level folder.
                                     This matters because BOTH botnets ship a
                                     file literally named 'scan.csv'.
      <device>/mirai_ack.csv         flat variant
    Returns None for stray files at the data root (N-BaIoT ships a
    header-only 'demonstrate_structure.csv' that must be ignored).
    """
    rel = path.relative_to(Path(DATA_ROOT))
    if len(rel.parts) < 2:
        return None                       # not inside a device folder -> skip

    device = rel.parts[0].replace('_', ' ')
    stem   = path.stem.lower()

    if any(k in stem for k in BENIGN_KEYS):
        return 0, 'benign', device

    botnet = next((p.lower() for p in rel.parts[1:-1]
                   if p.lower().startswith(BOTNET_KEYS)), '')
    attack = f'{botnet}_{stem}' if botnet else stem
    return 1, attack, device

def count_rows(path: Path) -> int:
    with open(path, 'rb') as fh:
        return max(sum(1 for _ in fh) - 1, 0)   # minus header

def read_sample(path: Path, n_rows: int, budget: int, seed: int) -> pd.DataFrame:
    if n_rows <= budget:
        df = pd.read_csv(path)
    else:
        rng = np.random.default_rng(seed)
        idx = np.arange(1, n_rows + 1)          # line 0 is the header
        keep = rng.choice(idx, size=budget, replace=False)
        skip = np.setdiff1d(idx, keep)          # sorted, required by pandas
        df = pd.read_csv(path, skiprows=skip.tolist())
    df.columns = [str(c).strip() for c in df.columns]
    return df

records = []
for p in sorted(Path(DATA_ROOT).rglob('*.csv')):
    cls = classify_file(p)
    if cls is None:
        continue                                # stray file at the data root
    records.append({'file': str(p), 'rows': count_rows(p),
                    'label': cls[0], 'attack': cls[1], 'device': cls[2]})

inv = pd.DataFrame(records)
inv = inv[inv['rows'] > 0].reset_index(drop=True)
assert len(inv) > 0, 'No usable CSV files found.'
print(f'{len(inv)} usable CSV files across {inv["device"].nunique()} devices')

# Proportional-but-balanced budget so tiny traffic types are not dropped
inv['budget'] = np.maximum(
    (inv['rows'] * TARGET_ROWS / inv['rows'].sum()).round().astype(int),
    min(500, int(inv['rows'].min())),
)

frames = []
for i, r in inv.iterrows():
    d = read_sample(Path(r['file']), int(r['rows']), int(r['budget']), SEED + i)
    d['device']  = r['device']
    d['attack']  = r['attack']
    d['label']   = r['label']
    frames.append(d)

df = pd.concat(frames, ignore_index=True)
print(f'Loaded {len(df):,} rows x {df.shape[1]} columns from {len(inv)} CSV files')


In [ ]:
# ---------- Step 1 deliverable: dataset summary ----------
META = ['device', 'attack', 'label']
FEATURES = [c for c in df.columns if c not in META]

summary = pd.DataFrame({
    'Field': ['Dataset name', 'Kaggle URL', 'Number of records (sampled)',
              'Number of records (files on disk)', 'Number of features',
              'Target variable', 'Devices', 'Attack types'],
    'Value': [
        'N-BaIoT Dataset to Detect IoT Botnet Attacks',
        'https://www.kaggle.com/datasets/mkashifn/nbaiot-dataset',
        f'{len(df):,}',
        f'{inv["rows"].sum():,}',
        f'{len(FEATURES)}',
        'label  (0 = benign, 1 = malicious)',
        f'{df["device"].nunique()}',
        f'{df["attack"].nunique() - 1}',
    ],
})
print(summary.to_string(index=False))
print()
print('Feature columns (first 15):', FEATURES[:15])

---
## §2  Define a Security Problem

### The problem

IoT devices are routinely compromised and conscripted into botnets. **Mirai** alone was used to launch some of the largest DDoS attacks on record, and it continues to resurface in variant form years after its source code leaked. The devices involved — doorbells, cameras, thermostats, baby monitors — share three properties that make them hard to defend:

1. **They cannot run endpoint security software.** Storage, memory and power budgets rule out host-based agents, and users rarely (if ever) patch them.
2. **They are physically exposed.** They sit on the same Wi-Fi as laptops and phones, and ship with default or hard-coded credentials.
3. **They are numerous and heterogeneous.** An enterprise may have dozens of device models, each with its own traffic fingerprint.

The only practical control that scales is **network-side monitoring**: observe a device's traffic from a gateway or IDS and flag behavior that deviates from its own baseline. If a compromised device can be identified *while it is attacking*, it can be quarantined in seconds — stopping the outbound flood and blocking further propagation — instead of days later.

### Research question

> **Can behavioral network-traffic features accurately distinguish benign IoT traffic from botnet-infected IoT traffic?**

Supporting questions:

1. Which attack types are hardest to detect — floods or low-and-slow scanning?
2. Which traffic features are most indicative of a compromised device?
3. What false-positive rate is operationally acceptable in a real detection system?

### Why this framing matters

A detector that is merely *accurate* is not automatically *useful*. In production, benign traffic vastly outnumbers attack traffic, so even a small false-positive rate can drown an analyst in alerts. Throughout §6 and §7 we therefore report **precision, recall, FPR and FNR explicitly**, and we translate the false-positive rate into a concrete operational figure (false alerts per million benign flows).

---
## §3  Data Exploration

We examine: class distribution · per-device composition · missing values · duplicate records · feature distributions · correlation structure · and which features separate the classes most strongly.

In [ ]:
# ---------- Basic shape, types, first look ----------
print('Shape:', df.shape)
print('Dtypes: numeric =', df[FEATURES].select_dtypes('number').shape[1],
      '| object =', df.select_dtypes('object').shape[1])
display(df.head(3))
print()
print('--- class distribution ---')
class_counts = df['label'].value_counts().sort_index()
class_counts.index = ['benign (0)', 'malicious (1)']
print(class_counts.to_string())
print(f"\nRatio malicious:benign = {class_counts.iloc[1]/class_counts.iloc[0]:.2f} : 1")

In [ ]:
# ---------- Class distribution + device composition ----------
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

counts = df['label'].value_counts().sort_index()
sns.barplot(x=['Benign', 'Malicious'], y=counts.values,
            palette=['#2ecc71', '#e74c3c'], ax=axes[0])
axes[0].set_title('Class distribution')
axes[0].set_ylabel('records')
for i, v in enumerate(counts.values):
    axes[0].text(i, v, f' {v:,}\n({v/len(df):.1%})', ha='center', va='bottom', fontsize=9)

ct = pd.crosstab(df['device'], df['label'])
ct.columns = ['Benign', 'Malicious'] if len(ct.columns) == 2 else ct.columns
ct = ct.sort_values(ct.columns[-1], ascending=True)
ct.plot(kind='barh', stacked=True, color=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_title('Records per device, by class')
axes[1].set_xlabel('records')
axes[1].legend(title='')

plt.tight_layout()
plt.show()

print('Attack types present:')
print(', '.join(sorted(a for a in df["attack"].unique() if a != 'benign')))

In [ ]:
# ---------- Missing values & duplicate records ----------
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if missing.empty:
    print('Missing values: none in any column.')
else:
    print('Missing values by column:')
    display(missing.to_frame('n_missing').assign(pct=lambda d: (d['n_missing']/len(df)).map('{:.2%}'.format)))
    fig, ax = plt.subplots(figsize=(8, max(3, 0.22*len(missing))))
    missing.head(30)[::-1].plot(kind='barh', ax=ax, color='#3498db')
    ax.set_title('Missing values (top 30 columns)')
    plt.tight_layout(); plt.show()

n_dup = int(df.duplicated().sum())
print(f'\nExact duplicate records: {n_dup:,}  ({n_dup/len(df):.2%} of the sample)')
if n_dup:
    by_cls = df[df.duplicated()]['label'].value_counts().sort_index()
    print('  of which benign:', int(by_cls.get(0, 0)), '| malicious:', int(by_cls.get(1, 0)))
print('Infinitely/NaN-producing values:',
      int(np.isinf(df[FEATURES].to_numpy(dtype=float, na_value=np.nan)).sum()))

In [ ]:
# ---------- Feature distributions ----------
# Pick the highest-variance features (they carry the signal) and show how
# benign vs malicious traffic differs on a log scale.
top_var = df[FEATURES].var().sort_values(ascending=False).head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.ravel(), top_var):
    for lab, colour, name in [(0, '#2ecc71', 'benign'), (1, '#e74c3c', 'malicious')]:
        s = df.loc[df['label'] == lab, col].dropna()
        s = np.log10(np.abs(s) + 1e-6)
        ax.hist(s, bins=60, alpha=0.55, color=colour, label=name, density=True)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('log10(|value| + 1e-6)')
    ax.legend(fontsize=8)
fig.suptitle('Distributions of the 6 highest-variance features (log scale)', y=1.01)
plt.tight_layout(); plt.show()

print('Summary statistics, top-variance features:')
display(df[top_var].describe().T.round(4))

In [ ]:
# ---------- Correlation analysis ----------
# 115 x 115 is unreadable, so we show the 15 highest-variance features,
# then prune the full matrix to uncorrelated representatives.
CORR_N = 15
corr_cols = df[FEATURES].var().sort_values(ascending=False).head(CORR_N).index

corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, mask=mask, cmap='vlag', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 7}, linewidths=0.3, ax=ax)
ax.set_title(f'Correlation matrix - {CORR_N} highest-variance features')
plt.tight_layout(); plt.show()

# How redundant is the whole feature set?
full_corr = df[FEATURES].corr().abs()
upper = full_corr.where(np.triu(np.ones(full_corr.shape), k=1).astype(bool))
print(f'Feature pairs with |r| > {CORR_THRESH}:  {int((upper > CORR_THRESH).sum().sum()):,}')
print(f'Mean |correlation| between features: {upper.stack().mean():.3f}')

In [ ]:
# ---------- Which features separate benign from malicious? ----------
# Standardised mean difference (Cohen's d style) - a simple, assumption-light
# effect size that ranks features by how far apart the classes sit.
g0 = df.loc[df['label'] == 0, FEATURES]
g1 = df.loc[df['label'] == 1, FEATURES]
pooled_sd = (g0.std() + g1.std()) / 2
cohens_d = ((g1.mean() - g0.mean()) / pooled_sd.replace(0, np.nan)).abs().sort_values(ascending=False)

top_sep = cohens_d.head(8).index.tolist()
print('Most discriminating features (|standardised mean difference|):')
print(pd.DataFrame({
    'benign_mean': g0[top_sep].mean().round(3),
    'malicious_mean': g1[top_sep].mean().round(3),
    'effect_size': cohens_d[top_sep].round(2),
}))

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), top_sep):
    tmp = df[['label', col]].copy()
    tmp[col] = np.sign(tmp[col]) * np.log10(np.abs(tmp[col]) + 1)
    sns.boxplot(data=tmp, x='label', y=col, palette=['#2ecc71', '#e74c3c'],
                showfliers=False, ax=ax)
    ax.set_xticklabels(['benign', 'malicious'])
    ax.set_title(col, fontsize=9)
    ax.set_ylabel('signed log10')
fig.suptitle('Top 8 separating features (signed log10 scale)', y=1.01)
plt.tight_layout(); plt.show()

### What we learn from the data

Recorded from the run above (150,304 rows sampled from 7,062,606):

- **Class balance.** N-BaIoT is unusual: **138,191 malicious (91.9%) against 12,113 benign (8.1%)** — a ratio of **11.4 : 1**. Attack records *outnumber* benign ones because the captures contain long flood and scan sessions. A production network is the exact opposite, where benign traffic dominates by orders of magnitude. This inverted prior is the largest single limitation of the study and we return to it in §7.
- **Device composition.** All nine devices contribute both classes, so no device is a proxy for the label — which is what allows a single pooled model to be evaluated fairly.
- **Missing values / duplicates.** **Zero missing values** across all 115 features, and **96 exact duplicate records (0.06%)** — 35 benign, 61 malicious. Duplicates come from flood generators emitting byte-identical flow snapshots; they are removed *before* the train/test split so they cannot leak across it.
- **Distribution shape.** Flow statistics are strongly right-skewed with extreme tails (`HH_jit_L0.01_variance` reaches 5.88e17), which is why log scales are used and why tree-based models were expected to outperform a plain logistic regression. They did, but only barely.
- **Correlation.** **334 feature pairs exceeded |r| > 0.95** (mean absolute correlation 0.272). The near-duplicate blocks arise because the same 23 statistics are repeated across five time windows — this motivates the feature pruning in §4.
- **Separating features.** The strongest effect sizes are all *weight* statistics, i.e. packet volume: `MI_dir_L0.1_weight` averages **14.5 on benign traffic versus 3,689.6 when infected — 254× more, d = 2.56**. Physically, a device that normally measures temperature or streams a doorbell view has started moving orders of magnitude more packets in a tenth of a second.

---
## §4  Data Preprocessing

Each step below is justified in-place. Order matters: **deduplicate before splitting** (so identical rows cannot leak across the split), and **fit every data-dependent transform on the training set only** (so the test set stays unseen).

In [ ]:
# ---------- 4.1 Remove duplicates ----------
# Why: N-BaIoT's flood generators emit identical flow snapshots repeatedly.
# Duplicates inflate accuracy (the model sees the same row in train and test)
# and waste memory, so they must go before any split.
n0 = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Removed {n0 - len(df):,} duplicate rows  ({(n0-len(df))/n0:.1%})  ->  {len(df):,} remain')

# ---------- 4.2 Train / test split (stratified on device x label) ----------
# Why: stratifying on the device-label combination guarantees every device and
# both classes appear in both partitions, so per-device results in 6 are valid.
# This is also the honest test: the model never sees these rows during training.
strata = df['device'].astype(str) + '|' + df['label'].astype(str)
strat = strata if strata.value_counts().min() >= 2 else df['label']

train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, random_state=SEED, stratify=strat,
)
print(f'Train {len(train_df):,}  |  Test {len(test_df):,}  ({TEST_SIZE:.0%})')
print('Train class mix:\n', train_df['label'].value_counts(normalize=True).round(4).to_string())

In [ ]:
# ---------- 4.3 Feature selection (variance + correlation pruning) ----------
# Why: 115 features include many near-constant ones and large blocks that are
# ~0.99 correlated (the same statistic at 100 ms / 500 ms / 1.5 s / 10 s / 1 min).
# Redundant features slow training, destabilise logistic regression coefficients,
# and dilute feature-importance rankings. Both filters are fitted on TRAIN only.
# NOTE: 'device' rides along because the ColumnTransformer one-hot encodes it;
# the numeric filters below are applied to the behavioural features only.
X_train_df = train_df[FEATURES + ['device']].copy()
X_test_df  = test_df[FEATURES + ['device']].copy()
num_only   = list(FEATURES)

# (a) drop near-zero-variance columns
var = X_train_df[num_only].var()
low_var = var[var < VARIANCE_EPS].index.tolist()
X_train_df = X_train_df.drop(columns=low_var)
X_test_df  = X_test_df.drop(columns=low_var)
num_only   = [c for c in num_only if c not in low_var]

# (b) greedily drop features that are > 0.95 correlated with an earlier feature
corr = X_train_df[num_only].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if (upper[c] > CORR_THRESH).any()]
X_train_df = X_train_df.drop(columns=to_drop)
X_test_df  = X_test_df.drop(columns=to_drop)
num_only   = [c for c in num_only if c not in to_drop]

kept = num_only
print(f'Features: {len(FEATURES)}  ->  {len(low_var)} removed (near-zero variance)  '
      f'->  {len(to_drop)} removed (|r| > {CORR_THRESH})  ->  {len(kept)} kept')

# ---------- 4.4 Encode categorical variables ----------
# Why: the device identity is a legitimate feature (each model has its own
# traffic fingerprint) but it is nominal, not ordinal - one-hot encoding avoids
# implying an ordering that does not exist.
cat_col = 'device'
num_cols = kept

# ---------- 4.5 Impute + scale ----------
# Why: median imputation is robust to the heavy right skew of flow statistics;
# scaling puts all features on one range so gradient-based models converge and
# distance-based interpretations are not dominated by a single large-magnitude
# feature. The scaler is fit on TRAIN ONLY to avoid leaking test statistics.
preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale',  StandardScaler()),
    ]), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), [cat_col]),
])

X_tr = preprocess.fit_transform(X_train_df)
X_te = preprocess.transform(X_test_df)
y_tr = train_df['label'].to_numpy()
y_te = test_df['label'].to_numpy()

print('X_train:', X_tr.shape, '| X_test:', X_te.shape)

# ---------- 4.6 Handle imbalanced classes ----------
# Why: N-BaIoT is imbalanced (and production traffic is far more so). Raw
# accuracy would reward a model that simply guesses the majority class, so all
# supervised models below use class_weight='balanced', which penalises errors on
# the minority class proportionally to its rarity.
weights = np.bincount(y_tr) / len(y_tr)
print(f'Class proportions in train: benign={weights[0]:.3f}  malicious={weights[1]:.3f}')
print('Mitigation: class_weight="balanced" on every supervised model; '
      'recall / F1 / FPR / FNR reported instead of accuracy alone.')

---
## §5  Security Analysis / Machine Learning

Four methods are applied, chosen to span complementary assumptions about what "abnormal" means:

| Method | Type | How it addresses the security problem |
|---|---|---|
| **Logistic Regression** | Supervised, linear | Interpretable baseline: each coefficient is a signed weight on a traffic statistic, so we can read *which* behaviours push a flow toward "malicious" and in which direction. |
| **Random Forest** | Supervised, ensemble | Captures non-linear and interaction effects (e.g. jitter only matters when packet rate is also high) and gives a built-in feature-importance ranking. Robust to unscaled, skewed input. |
| **XGBoost** | Supervised, gradient boosting | State-of-the-art tabular learner; usually the strongest accuracy and the best-calibrated scores, and it exposes SHAP values for explainability (§8). |
| **Isolation Forest** | **Unsupervised anomaly detection** | Trained on **benign traffic only**, it never sees an attack. It flags statistically *unusual* flows as anomalies - the closest analogue to a real IDS that cannot enumerate every future attack signature, and it can therefore detect novel/increasingly evasive behaviour. |

**Important:** Isolation Forest is fit on benign training rows only (`y == 0`). That is what makes it a genuine anomaly detector rather than a classifier in disguise.

In [ ]:
# ============================================================
#  Train all four models and score them on the held-out test set
# ============================================================
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=2000, class_weight='balanced', random_state=SEED),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, n_jobs=-1, class_weight='balanced', random_state=SEED),
    'XGBoost': XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.9, colsample_bytree=0.9,
        eval_metric='logloss', random_state=SEED, n_jobs=-1,
        verbosity=0),
    'Isolation Forest': IsolationForest(
        n_estimators=300, contamination='auto', random_state=SEED, n_jobs=-1),
}

def score_of(name, model, X):
    """Return (hard predictions, anomaly/positive score in [0,1] direction)."""
    if name == 'Isolation Forest':
        pred = (model.predict(X) == -1).astype(int)      # -1 = outlier = malicious
        score = -model.score_samples(X)                 # higher = more anomalous
        score = (score - score.min()) / (score.max() - score.min() + 1e-12)
    else:
        pred = model.predict(X)
        score = model.predict_proba(X)[:, 1]
    return pred, score

results, preds, scores = {}, {}, {}
for name, model in models.items():
    if name == 'Isolation Forest':
        model.fit(X_tr[y_tr == 0])                       # benign only!
    else:
        model.fit(X_tr, y_tr)
    pred, score = score_of(name, model, X_te)
    preds[name], scores[name] = pred, score
    tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
    results[name] = {
        'Accuracy':  accuracy_score(y_te, pred),
        'Precision': precision_score(y_te, pred, zero_division=0),
        'Recall':    recall_score(y_te, pred, zero_division=0),
        'F1':        f1_score(y_te, pred, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_te, score),
        'FPR':       fp / (fp + tn),
        'FNR':       fn / (fn + tp),
        'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn,
    }
    print(f'trained: {name}')

print('\nDone.')

---
## §6  Model Evaluation

Metrics reported: **Accuracy · Precision · Recall · F1 · ROC-AUC · Confusion Matrix · FPR · FNR**.

In [ ]:
# ---------- Overall metrics ----------
res = pd.DataFrame(results).T
res = res[['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'FPR', 'FNR',
           'TP', 'FP', 'TN', 'FN']]
display(res.round(4))

# LaTeX table ready to paste into the Overleaf report
print('--- LaTeX ---')
print(res[['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'FPR', 'FNR']]
      .round(4).to_latex(escape=False,
      caption='Overall model performance on the held-out test set.',
      label='tab:overall'))

best = res['F1'].idxmax()
print('\nBest F1:', best, '| Best ROC-AUC:', res['ROC-AUC'].idxmax())

In [ ]:
# ---------- Confusion matrices ----------
fig, axes = plt.subplots(1, len(models), figsize=(4*len(models), 3.8))
for ax, (name, pred) in zip(axes, preds.items()):
    cm = confusion_matrix(y_te, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['benign', 'malicious'],
                yticklabels=['benign', 'malicious'], ax=ax)
    ax.set_xlabel('predicted'); ax.set_ylabel('actual')
    ax.set_title(name, fontsize=11)
plt.tight_layout(); plt.show()

print('Normalized (row %):')
for name, pred in preds.items():
    cm = confusion_matrix(y_te, pred, normalize='true')
    print(f'  {name:22s} -> benign recalled {cm[0,0]:.4f} | malicious recalled {cm[1,1]:.4f}')

In [ ]:
# ---------- ROC curves ----------
fig, ax = plt.subplots(figsize=(7.5, 6))
colours = ['#3498db', '#e67e22', '#9b59b6', '#16a085']
for (name, score), c in zip(scores.items(), colours):
    fpr, tpr, _ = roc_curve(y_te, score)
    ax.plot(fpr, tpr, lw=2, color=c,
            label=f'{name} (AUC = {results[name]["ROC-AUC"]:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC curves - benign vs malicious IoT traffic')
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ---------- Per-device performance ----------
# A detector that only works on one camera is not deployable, so we break the
# metrics out by device to check generalisation across the fleet.
per_dev = []
dev_true = test_df['device'].to_numpy()
for name, pred in preds.items():
    for d in pd.unique(dev_true):
        m = dev_true == d
        if m.sum() == 0 or len(np.unique(y_te[m])) < 2:
            continue
        tn, fp, fn, tp = confusion_matrix(y_te[m], pred[m]).ravel()
        per_dev.append({
            'Model': name, 'Device': d, 'n': int(m.sum()),
            'Accuracy': accuracy_score(y_te[m], pred[m]),
            'Recall':   recall_score(y_te[m], pred[m], zero_division=0),
            'F1':       f1_score(y_te[m], pred[m], zero_division=0),
            'FPR':      fp/(fp+tn), 'FNR': fn/(fn+tp),
        })
per_dev = pd.DataFrame(per_dev)

pivot = per_dev.pivot(index='Device', columns='Model', values='F1').round(4)
display(pivot)
print('--- LaTeX (F1 by device) ---')
print(pivot.to_latex(escape=False, index_names=False,
      caption='F1-score per device.', label='tab:per_device'))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
per_dev.pivot(index='Device', columns='Model', values='FNR').plot(
    kind='barh', ax=axes[0], color=colours)
axes[0].set_title('False Negative Rate by device (missed attacks)')
axes[0].set_xlabel('FNR')
per_dev.pivot(index='Device', columns='Model', values='FPR').plot(
    kind='barh', ax=axes[1], color=colours)
axes[1].set_title('False Positive Rate by device (false alarms)')
axes[1].set_xlabel('FPR')
plt.tight_layout(); plt.show()

In [ ]:
# ---------- Per-attack-type detection ----------
# Which attacks are easy, and which slip through? This is the question a
# security team actually cares about.
attack_true = test_df['attack'].to_numpy()
attack_tbl = []
for name, pred in preds.items():
    for a in pd.unique(attack_true):
        m = attack_true == a
        if m.sum() == 0:
            continue
        # for a benign slice, "recall of the benign class" = 1 - FPR on benign
        sub_pred, sub_true = pred[m], y_te[m]
        if a == 'benign':
            val, kind = float((sub_pred == 0).mean()), 'specificity'
        else:
            val, kind = float((sub_pred == 1).mean()), 'recall'
        attack_tbl.append({'Model': name, 'Attack': a, 'n': int(m.sum()), kind: val})
attack_df = pd.DataFrame(attack_tbl)
attack_df = attack_df.melt(id_vars=['Model', 'Attack', 'n'],
                           var_name='metric', value_name='value').dropna()

best_models = ['XGBoost', 'Random Forest', 'Isolation Forest']
sub = attack_df[attack_df['Model'].isin(best_models)]
piv = sub.pivot_table(index='Attack', columns='Model', values='value').round(4)
display(piv)
print('--- LaTeX (detection rate by attack type) ---')
print(piv.to_latex(escape=False, index_names=False,
      caption=('Detection rate by attack type. Attack rows = recall '
               '(1.0000 = every instance caught); the benign row = specificity.'),
      label='tab:attacks'))

piv.plot(kind='barh', figsize=(9, 6), color=colours[:3])
plt.xlabel('detection rate'); plt.title('Detection rate by attack type')
plt.xlim(0, 1.05); plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

In [ ]:
# ---------- Operational translation of the false-positive rate ----------
# Security teams care about alert volume, not just FPR. We translate each
# model's FPR into "false alarms per million benign flows" and its FNR into
# "missed attacks per million attack flows" - two different denominators.
BENIGN_FLOWS_PER_DAY = 1_000_000   # illustrative enterprise gateway volume
ATTACK_FLOWS_PER_DAY =    10_000   # illustrative attack traffic per day

ops = pd.DataFrame(results).T[['FPR', 'FNR', 'Precision', 'Recall']].copy()
ops['False alerts / 1M benign flows / day'] = ops['FPR'] * BENIGN_FLOWS_PER_DAY
ops['Missed attacks / day']                 = ops['FNR'] * ATTACK_FLOWS_PER_DAY
ops = ops.round(4)
display(ops)
print('--- LaTeX ---')
print(ops.to_latex(escape=False, index_names=False,
      caption=(f'Operational impact at {BENIGN_FLOWS_PER_DAY:,} benign flows '
               f'and {ATTACK_FLOWS_PER_DAY:,} attack flows per day.'),
      label='tab:ops'))

---
## §7  Security Interpretation

> **Results this section is based on** — sampled 150,304 of **7,062,606** records; 115 → **36** features after pruning; 120,166 train / 30,042 test (**2,416 benign / 27,626 malicious**).

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC | FPR | FNR |
|---|---|---|---|---|---|---|---|
| Logistic Regression | 0.9992 | 0.9996 | 0.9996 | 0.9996 | 0.9989 | 0.0046 | 0.0004 |
| Random Forest | 0.9999 | 1.0000 | 0.9999 | 1.0000 | 1.0000 | 0.0000 | 0.0001 |
| **XGBoost** | **1.0000** | **1.0000** | **1.0000** | **1.0000** | **1.0000** | **0.0000** | **0.0000** |
| Isolation Forest | 0.2978 | 0.9706 | **0.2438** | 0.3897 | 0.8835 | **0.0844** | **0.7562** |

### 7.1 What the detector actually detects

The models were not trained on "attacks" in the abstract — they separate **benign traffic from traffic produced by devices running Mirai and BASHLITE (Gafgyt)**. Concretely, they detect a device that is *scanning for new victims*, *flooding a target*, or *transmitting junk data on command*.

Looking at *why* they succeed is more instructive than the accuracy figure. The effect-size ranking in §3 is unambiguous:

| Feature | benign mean | malicious mean | ratio | effect size |
|---|---|---|---|---|
| `MI_dir_L0.1_weight` | 14.5 | **3,689.6** | **×254** | 2.56 |
| `MI_dir_L1_weight` | 6.4 | **398.0** | ×62 | 2.37 |
| `MI_dir_L0.01_weight` | 68.9 | **21,472.4** | ×311 | 2.22 |
| `MI_dir_L3_weight` | 4.8 | **139.9** | ×29 | 2.21 |

`weight` is *"the number of items observed in recent history"* — i.e. **packet volume**. In the shortest windows (0.01 s and 0.1 s), an infected device moves **two to three orders of magnitude more packets** than it does when behaving normally. The detector is therefore not performing malware analysis; it is recognising that **a device whose job is to measure temperature or stream a doorbell view has suddenly started generating traffic at flood rates**.

This is exactly the property that makes IoT botnet detection tractable: task-oriented devices have narrow, predictable behaviour, so the deviation is enormous rather than subtle.

### 7.2 Which attacks are difficult to detect

This is the most important result in the project, and it splits sharply by method.

**Supervised models — essentially nothing is missed.** Random Forest and XGBoost scored **1.0000 recall on 9 of the 10 attack types**; the only sub-perfect cells were `gafgyt_tcp` (RF 0.9997) and `gafgyt_udp` (RF/XGB 0.9998). Logistical Regression reached 0.9996 recall with 12 false negatives.

**Unsupervised anomaly detection — fails on exactly the attacks that matter most:**

| Attack | Isolation Forest | RF | XGBoost |
|---|---|---|---|
| `mirai_udpplain` | 0.6523 | 1.0000 | 1.0000 |
| `mirai_udp` | 0.6514 | 1.0000 | 1.0000 |
| `mirai_ack` | 0.6491 | 1.0000 | 1.0000 |
| `gafgyt_junk` | **0.0274** | 1.0000 | 1.0000 |
| `gafgyt_scan` | **0.0226** | 1.0000 | 1.0000 |
| `gafgyt_combo` | **0.0024** | 1.0000 | 1.0000 |
| `gafgyt_udp` | **0.0002** | 0.9998 | 0.9998 |
| `mirai_scan` | **0.0000** | 1.0000 | 1.0000 |
| `mirai_syn` | **0.0000** | 1.0000 | 1.0000 |
| `gafgyt_tcp` | **0.0000** | 0.9997 | 1.0000 |

Two things follow.

1. **Scanning is the hard case, and it is the dangerous one.** Isolation Forest detects `mirai_scan` and `mirai_syn` **zero percent** of the time, and `gafgyt_scan` at 2.3%. Scanning is the **propagation** phase — a miss means the botnet keeps recruiting devices and the defender's blast radius keeps growing. Flooding is loud but self-limiting; scanning is quiet and compounding.

2. **"Anomaly detection" is not automatically good at detecting attacks.** The Isolation Forest only caught the three highest-rate Mirai floods (≈0.65) and missed *both* low-and-slow botnets almost entirely. The reason is structural: benign IoT traffic is itself highly variable — a camera streaming video produces enormous packet volumes perfectly legally — so *high volume alone is not anomalous*. Anything that merely looks like a busy device blends in. Its overall **FNR of 0.7562 means three out of every four attack instances went undetected**, at an overall accuracy of 0.2978.

   This also explains the gap with the original N-BaIoT paper, which reported 100% TPR. That result used **one deep autoencoder per device**, with a threshold and voting window tuned on a held-out benign set. We used **one pooled Isolation Forest with default hyperparameters**. The comparison is the security lesson: anomaly detection is extremely sensitive to per-device baselining and threshold tuning, and a naively configured detector can be worse than useless.

3. Supervised near-perfection should be read cautiously — see §7.7.

### 7.3 Which features are most important

**XGBoost gain importance (top 10 of 36):**

| Rank | Feature | Importance |
|---|---|---|
| 1 | `HpHp_L0.01_radius` | **0.6898** |
| 2 | `HH_L0.01_pcc` | **0.1829** |
| 3 | `HH_jit_L5_mean` | 0.0442 |
| 4 | `HH_L0.01_weight` | 0.0255 |
| 5 | `MI_dir_L0.01_weight` | 0.0210 |
| 6 | `HH_L0.01_covariance` | 0.0188 |
| 7 | `HH_L0.01_radius` | 0.0093 |
| 8 | `HH_jit_L1_variance` | 0.0014 |
| 9 | `MI_dir_L5_weight` | 0.0014 |
| 10 | `HH_jit_L0.1_variance` | 0.0007 |

Two features carry **87% of the total gain**, and the vocabulary of the top of the list is consistent:

- **`L0.01` dominates** — that is the *shortest* temporal window (10 ms). Detection is driven by what happens in the **most recent few milliseconds**, not by long-run averages.
- **`HpHp` = socket level** (source IP+port → destination IP+port) and **`HH` = channel level**. `radius` is the root-sum-of-squares of the two streams' variances — a direct measure of **burstiness on a single connection**. Attack floods are extremely bursty in a way that normal request/response traffic is not.
- **`HH_jit` = channel jitter.** Flood generators and C&C beacons are mechanically timed; real sensors are not. Note that `HH_jit_L5_mean` was also the single largest SHAP contributor on the example malicious flow (**+6.75**, versus all other features ≤ +0.54).

**SHAP on individual predictions** confirms the picture and shows both directions agree:

```
MALICIOUS (p = 1.0000)          BENIGN (p = 0.0000)
 HH_jit_L5_mean    +6.75         MI_dir_L0.01_weight  -5.05
 HH_L0.01_radius   +0.54         HH_jit_L5_mean       -2.50
 MI_dir_L0.01_weight+0.52        HpHp_L0.01_radius    -1.64
 HpHp_L0.01_radius +0.51         HH_L0.01_radius      -0.92
 HH_L0.01_std      +0.35         HH_jit_L0.1_variance -0.73
```

**Operational reading:** the model is deciding on *burstiness on a single socket within the last 10 ms*, corroborated by *jitter over the last 5 s* and *packet volume over the last 10 ms*. That is a feature set cheap and fast enough to compute incrementally at a gateway — which is precisely what a real-time IDS requires.

### 7.4 Consequences of false positives

A false positive labels a **legitimate device as compromised**.

**Measured:** Logistic Regression produced **11 false positives** out of 2,416 benign test flows → FPR = 0.0046. Random Forest and XGBoost produced **0**. Isolation Forest produced **204** → FPR = 0.0844.

Scaled to an illustrative gateway handling 1,000,000 benign flows/day:

| Model | FPR | False alerts / day |
|---|---|---|
| Logistic Regression | 0.0046 | **4,553** |
| Random Forest | 0.0000 | 0 |
| XGBoost | 0.0000 | 0 |
| Isolation Forest | 0.0844 | **84,437** |

- **Alert fatigue.** 4,553 alerts/day is roughly two per minute — no human team can triage it, and the tool gets switched off. 84,437/day is not a security product, it is a denial-of-service against your own SOC.
- **Availability and safety.** If the response is automatic quarantine, a false positive disconnects a camera, doorbell or thermostat. For a baby monitor or a security camera this is a **physical-safety and privacy outage**, not an inconvenience — and unlike an IT ticket, it cannot simply be reverted from a desk.
- **Why "0" is not really 0 — an important caveat.** Only **2,416 benign flows** landed in the test set. A measured FPR of 0 over 2,416 trials does not establish a zero false-positive rate: by the rule of three, the 95% confidence upper bound is 3/2416 ≈ **0.124%**, i.e. the true rate could still be as high as **~1,242 false alerts/day** at 1M flows. **The benign class is too small to certify the low FPR that deployment requires.** Any real evaluation would need millions of benign flows.

### 7.5 Consequences of false negatives

A false negative means a compromised device **stays on the network**.

**Measured:** Logistic Regression FN = 12, Random Forest FN = 2, XGBoost FN = 0, Isolation Forest FN = **20,890** (FNR = 0.7562) — at 10,000 attack flows/day that is **~7,562 missed attacks per day**.

The consequences compound:

1. **Ongoing DDoS participation.** The device keeps attacking third parties. Liability and reputational damage fall on *the network owner*, not on the device vendor.
2. **Continued propagation.** Missed **scanning** (0.0000 recall for Isolation Forest on `mirai_scan`) means the botnet keeps recruiting — the failure mode grows over time instead of being bounded.
3. **Persistent inside foothold.** The attacker retains an on-network host for lateral movement and data exfiltration, and the compromise is never surfaced to incident response.
4. **Privacy/surveillance breach.** In a home or office, an infected camera is being watched by someone else.

**Why FNs dominate the decision:** a false positive costs a device going offline until a human clears it; a false negative costs an active, ongoing attack originating from inside your perimeter. This asymmetry is why **recall and FNR — not accuracy — are treated as the primary measures here**, even though the choice makes Logistic Regression look worse than its 0.9992 accuracy suggests.

### 7.6 Use in a real security system

- **Placement.** On the **Wi-Fi access point or edge gateway**, where all device traffic already transits. No agent, no patch, no vendor cooperation, no drain on the device — which is the entire reason network-side detection is the right choice for IoT.
- **Granularity.** Per-device results show the pooled model already generalises: **XGBoost F1 = 1.0000 on 8 of 9 devices and 0.9999 on Ecobee**; Random Forest ≥ 0.9998 everywhere; Logistic Regression dips to 0.9982 on the Samsung webcam. Isolation Forest, by contrast, collapses on two devices (**F1 = 0.0015 on both Ennio Doorbell and Samsung Webcam** — essentially no better than random) while reaching only 0.27–0.59 on the rest. **Conclusion: supervised pooling works; unsupervised detection must be per-device.**
- **Response tiers.** High-confidence predictions (score > 0.99) → automatic quarantine. Medium → analyst review. Thresholds should be chosen from the FPR/FNR trade-off in §6, not defaulted to 0.5 — and set against a realistic benign class prior.
- **Re-baselining.** Firmware updates change traffic patterns; models need periodic retraining on fresh benign data, or the FPR will drift upward.
- **Feature budget.** The top-10 features above are all simple incremental statistics (counts, variances, jitter) over 10 ms–5 s windows — cheap enough for continuous scoring on commodity gateway hardware.
- **Ensemble strategy.** Use the supervised model as the primary detector and reserve anomaly detection as a *secondary, per-device* signal for behaviour outside the labelled attack repertoire.

### 7.7 Limitations

1. **Lab versus production.** N-BaIoT is nine devices, two botnets, one isolated network, one capture period. Real traffic adds protocols, encryption, far more benign variety, and devices whose behaviour changes over time.
2. **Inverted class prior — the single biggest caveat.** The corpus is **92% malicious** (7.87% benign). A production network is the opposite. Precision of 1.0000 measured here would **collapse** under a realistic prior: with FPR 0.0046 and a 0.1% attack rate, the vast majority of "alerts" would be benign traffic, not attacks. Every precision figure in §6 is therefore optimistic.
3. **Benign test set too small to certify FPR.** 2,416 benign samples. Zero observed false positives only bounds the true rate to ≲0.124% (95% CI) — up to ~1,242 false alerts/day. See §7.4.
4. **Random rather than chronological split.** Rows are split randomly even though consecutive flow snapshots come from the same capture session. Near-duplicates across the split inflate results. A time-based split is the honest evaluation and is not what was done.
5. **Duplicates were removed but only 96 existed (0.06%)** — so leakage risk from exact duplicates is low, but *near*-duplicates (adjacent snapshots) were not addressed.
6. **No adversarial testing.** An attacker who knows detection keys on socket burstiness and jitter can rate-limit and pad timing to evade it. Robustness to an adaptive adversary is entirely unmeasured — and a capable attacker would do exactly this.
7. **Two botnet families, and attack traffic is unusually "clean".** Mirai and BASHLITE floods are not subtle. Novel malware that mimics legitimate device behaviour is precisely what a supervised model will miss; the unsupervised alternative tested here demonstrably cannot fill that gap either (§7.2).
8. **Content-blind.** The 115 features abstract away payload, so the detector can say *behaviour deviates* but never *what* was sent — it cannot distinguish a DDoS from legitimate high-volume streaming on its own, which is why false positives carry such weight.
9. **Isolation Forest configuration is naive.** Default `contamination='auto'`, one pooled model, no threshold tuning, no per-device baseline, no majority-vote window. Its poor showing is a property of *this configuration*, not a verdict on anomaly detection — the original paper's tuned per-device autoencoders reached 100% TPR.
10. **Offline only.** No streaming feature extraction or detection-latency measurement; the original study reported ~174 ms to detect, which is the number a deployable IDS must hit.



---
## §8  Optional Advanced Component — Feature Importance & SHAP

**SHAP (SHapley Additive exPlanations)** assigns each prediction a per-feature contribution, giving a model-independent explanation of *why* a flow was flagged. For a security tool this is not a nicety: an analyst who is told *"flagged because packet-jitter variance at 1 min is 4.2σ above baseline"* can act, whereas a bare probability cannot be defended in a post-incident review.

In [ ]:
# ---------- Built-in feature importance (Random Forest & XGBoost) ----------
rf  = models['Random Forest']
xgb = models['XGBoost']

feat_names = list(preprocess.get_feature_names_out())
feat_names = [f.replace('num__', '').replace('cat__', '') for f in feat_names]

imp = pd.DataFrame({
    'feature': feat_names,
    'Random Forest': rf.feature_importances_,
    'XGBoost': xgb.feature_importances_,
}).set_index('feature')

for col in imp.columns:
    top = imp[col].sort_values(ascending=False).head(15)[::-1]
    top.plot(kind='barh', figsize=(8, 5), color='#2980b9' if col == 'Random Forest' else '#e67e22')
    plt.title(f'Top 15 features - {col}')
    plt.xlabel('importance'); plt.tight_layout(); plt.show()

print('--- LaTeX (XGBoost top 15) ---')
print(imp['XGBoost'].sort_values(ascending=False).head(15).round(4)
      .rename('importance').to_latex(escape=False, index_names=False,
      caption='XGBoost feature importance (top 15).', label='tab:importance'))

In [ ]:
# ---------- SHAP summary plot ----------
import shap

# Subsample the test set: SHAP on the full set is slow and visually identical
SHAP_N = min(2000, X_te.shape[0])
rng = np.random.default_rng(SEED)
idx = rng.choice(X_te.shape[0], size=SHAP_N, replace=False)
X_shap = X_te[idx]

explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_shap)

print(f'SHAP computed on {SHAP_N:,} test rows | matrix shape {np.shape(shap_values)}')

shap.summary_plot(shap_values, features=X_shap, feature_names=feat_names,
                  max_display=20, show=True)

In [ ]:
# ---------- SHAP: single-flow explanation (what an analyst would see) ----------
# Show one benign and one malicious test row so the explanation can be compared.
# matplotlib=True is used deliberately: it renders statically and works in
# Colab, Kaggle, Jupyter and plain scripts alike (no shap.initjs() needed).
expl = shap.TreeExplainer(xgb)

mal_i = int(np.where(y_te == 1)[0][0])
ben_i = int(np.where(y_te == 0)[0][0])

for label, i in [('MALICIOUS', mal_i), ('BENIGN', ben_i)]:
    v = X_te[i:i+1]
    sv = expl.shap_values(v)[0]
    base = float(np.array(expl.expected_value).reshape(-1)[0])
    pred = float(xgb.predict_proba(v)[0, 1])
    print(f'\n{label}: model probability = {pred:.4f}  | base value = {base:.4f}')
    order = np.argsort(np.abs(sv))[::-1][:5]
    for j in order:
        print(f'   {feat_names[j]:>40s}  {sv[j]:+.4f}')
    shap.force_plot(base, sv, features=v, feature_names=feat_names, matplotlib=True)
    plt.show()

---
## §9  Conclusion

**Answer to the research question:**

> **Yes — but with important caveats.** Behavioural network-traffic features distinguish benign from botnet-infected IoT traffic almost perfectly on this benchmark: **XGBoost F1 = 1.0000 and ROC-AUC = 1.0000**, Random Forest F1 = 1.0000, Logistic Regression F1 = 0.9996 — consistently across **all nine devices**.

**Security takeaways:**

1. **Network-side detection is viable for devices that cannot protect themselves.** No agent, no patch, no vendor cooperation — the deployment constraint that defines IoT security is satisfied by construction.
2. **The signal is volume and burstiness, not malware content.** Infected devices move **100–300× more packets in the 10 ms window** than they do when benign, and the top two features (`HpHp_L0.01_radius`, `HH_L0.01_pcc`) carry **87% of XGBoost's total gain**. Task-oriented devices are predictable, so the deviation is enormous rather than marginal.
3. **Floods are easy; scanning is the weak point — and it matters most.** Supervised models caught 9 of 10 attack types with 1.0000 recall, but **Isolation Forest detected `mirai_scan` and `mirai_syn` 0% of the time** and `gafgyt_scan` 2.3%. Scanning is the propagation phase, so those are the misses that let a botnet grow. Unsupervised detection overall missed **3 of every 4 attack instances** (FNR = 0.7562).
4. **Accuracy is the wrong headline metric for security.** Isolation Forest's 84,437 false alerts/day and 7,562 missed attacks/day sit behind a single accuracy number; Logistic Regression's 0.9992 accuracy conceals 4,553 false alerts/day. The error *rates* and their asymmetric costs are what an operator actually experiences.

**Honest caveats (§7.7):** the corpus is 92% attack traffic so every precision figure is optimistic; only 2,416 benign test flows mean the measured "zero" FPR only bounds the true rate to ≲0.124%; the split was random rather than chronological; and there was no adversarial testing.

**Future work:** chronological splitting to remove temporal leakage, adversarial robustness evaluation against an adaptive attacker, per-device tuned anomaly detectors to close the gap with the paper's autoencoders, and a streaming harness to measure detection latency.

